# ForensicTAB: baselines

## Install and import

In [ ]:
!pip -q install scikit-multilearn iterative-stratification openpyxl

In [ ]:
import os, re, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score, jaccard_score,
                             hamming_loss, roc_auc_score, roc_curve, auc, average_precision_score,
                             precision_recall_curve, multilabel_confusion_matrix, classification_report)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
    try: keras.utils.set_random_seed(s)
    except Exception: pass
set_seed()

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

##  Configuration

In [ ]:
OPTIMIZATION_MODE = "quick"        # "full" | "quick" | "skip"
USE_AUGMENTATION  = False          # paper uses none; True enables categorical feature-masking augmentation
AUG_MASK_PROB, AUG_COPIES = 0.10, 1
THRESHOLD = 0.5                    # fixed decision threshold (never tuned on val/test)
RUN_REPEATED_SEEDS = False         # optional: retrain with seeds 0-9 and report mean ± SD

LABELS   = ["WhatsApp Chats", "SMS Messages", "Call Logs", "Contacts", "Media Files", "Browser History"]
FEATURES = ["Device Make", "Device Model", "OS Type", "OS Version", "Extraction Tool", "Tool Version", "Extraction Method"]

# Alternative column spellings found in the spreadsheet (case / spaces / punctuation are ignored when matching)
ALIASES = {
    "Device Make": ["Make", "Manufacturer", "Device Manufacturer", "Brand"],
    "Device Model": ["Model"],
    "OS Type": ["OS", "Operating System", "OS Name"],
    "OS Version": ["Android Version", "OS Ver"],
    "Extraction Tool": ["Tool", "Forensic Tool", "Acquisition Tool"],
    "Tool Version": ["Tool Ver", "Version of Tool"],
    "Extraction Method": ["Method", "Acquisition Method", "Extraction Type"],
    "WhatsApp Chats": ["WhatsApp", "Whatsapp Chat"],
    "SMS Messages": ["SMS", "Messages"],
    "Call Logs": ["Call Log", "Calls"],
    "Contacts": ["Contact"],
    "Media Files": ["Media", "Multimedia"],
    "Browser History": ["Browser", "Browsing History"],
}
# Used only if the six label columns are NOT present and must be parsed from the free-text field
TEXT_FIELD_ALIASES = ["Recovered Artifacts", "Recovered Artefacts", "Artifacts Recovered"]
KEYWORDS = {
    "WhatsApp Chats":  r"whats\s*app",
    "SMS Messages":    r"\bsms\b|text message",
    "Call Logs":       r"call\s*log|call\s*history|\bcalls?\b",
    "Contacts":        r"contact",
    "Media Files":     r"media|image|photo|picture|video|audio|multimedia",
    "Browser History": r"browser|browsing|web\s*history",
}

# Paper's final configuration (Table 1)
PAPER_PARAMS = dict(emb=32, heads=4, layers=2, dff=128, kan=64, dropout=0.30,
                    lr=1e-4, batch=16, epochs=100, patience=15)

##  Load dataset

In [ ]:
def upload_dataset():
    try:                                              # Google Colab
        from google.colab import files
        up = files.upload()
        return next(iter(up))
    except ImportError:
        pass
    try:                                              # local Jupyter with a display
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        path = filedialog.askopenfilename(title="Select the forensic dataset",
                                          filetypes=[("Data files", "*.csv *.xlsx *.xls"), ("All files", "*.*")])
        root.destroy()
        if path: return path
    except Exception:
        pass
    return input("Enter the full path to the dataset file: ").strip().strip('"')

DATA_PATH = upload_dataset()
print("Selected:", DATA_PATH)

Saving archive (2) (1).zip to archive (2) (1).zip
Selected: archive (2) (1).zip


In [ ]:
def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())

def find_col(name, df, extra=()):
    lookup = {norm(c): c for c in df.columns}
    for cand in [name, *ALIASES.get(name, []), *extra]:
        if norm(cand) in lookup: return lookup[norm(cand)]
    return None

feat_cols = {f: find_col(f, raw) for f in FEATURES}
lab_cols  = {l: find_col(l, raw) for l in LABELS}
print("Feature columns :", feat_cols)
print("Label columns   :", lab_cols)
missing_f = [f for f, c in feat_cols.items() if c is None]
assert not missing_f, f"Feature columns not found: {missing_f}. Add spellings to ALIASES."

POS = {"1", "1.0", "yes", "y", "true", "t", "recovered", "available", "present", "✓", "✔"}
def to_bin(v):
    if pd.isna(v): return 0
    return int(str(v).strip().lower() in POS)

df = raw.copy()
if all(c is not None for c in lab_cols.values()):
    for l, c in lab_cols.items(): df[l] = raw[c].map(to_bin)
    print("Targets taken from the six label columns.")
else:
    txt = find_col(TEXT_FIELD_ALIASES[0], raw, extra=TEXT_FIELD_ALIASES[1:])
    assert txt is not None, "Neither the six label columns nor a 'Recovered Artifacts' text column was found."
    print(f"Targets parsed from the free-text column '{txt}'")
    s = raw[txt].fillna("").astype(str).str.lower()
    for l in LABELS: df[l] = s.str.contains(KEYWORDS[l], regex=True).astype(int)

df = df.drop_duplicates().reset_index(drop=True)                       # exact duplicate records only
print("After removing exact duplicates:", df.shape)


Feature columns : {'Device Make': 'Device Make', 'Device Model': 'Model', 'OS Type': 'OS Type', 'OS Version': 'OS Version', 'Extraction Tool': 'Extraction Tool', 'Tool Version': 'Tool Version', 'Extraction Method': 'Extraction Method'}
Label columns   : {'WhatsApp Chats': 'WhatsApp Chats', 'SMS Messages': 'SMS Messages', 'Call Logs': 'Call Logs', 'Contacts': 'Contacts', 'Media Files': 'Media Files', 'Browser History': 'Browser History'}
Targets taken from the six label columns.
After removing exact duplicates: (1500, 17)


## 4. Pre-processing, iterative stratified split, optional augmentation

In [ ]:
PLACEHOLDERS = {"", "nan", "none", "-", "n/a", "na", "null", "unknown", "not available"}

def clean_feature(s):
    s = s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    s = s.where(~s.str.lower().isin(PLACEHOLDERS), "Unknown")
    canon = {}                                                       #
    for v in s: canon.setdefault(v.lower(), v)
    return s.str.lower().map(canon)

# Map each feature using its matched column name from feat_cols
X_df = pd.DataFrame({f: clean_feature(df[feat_cols[f]]) for f in FEATURES})
Y = df[LABELS].values.astype("float32")
n = len(X_df)

def stratified_split(Y, n_test, n_val, seed=SEED):
    idx = np.arange(len(Y)).reshape(-1, 1)
    try:
        from skmultilearn.model_selection import iterative_train_test_split
        np.random.seed(seed)
        i_tmp, y_tmp, i_te, y_te = iterative_train_test_split(idx, Y, test_size=n_test / len(Y))
        i_tr, y_tr, i_va, y_va = iterative_train_test_split(i_tmp, y_tmp, test_size=n_val / len(i_tmp))
        return i_tr.ravel(), i_va.ravel(), i_te.ravel(), "skmultilearn iterative stratification"
    except Exception as e1:
        try:
            from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
            s1 = MultilabelStratifiedShuffleSplit(1, test_size=n_test / len(Y), random_state=seed)
            tmp, te = next(s1.split(idx, Y))
            s2 = MultilabelStratifiedShuffleSplit(1, test_size=n_val / len(tmp), random_state=seed)
            tr_i, va_i = next(s2.split(idx[tmp], Y[tmp]))
            return tmp[tr_i], tmp[va_i], te, "iterative-stratification (MultilabelStratifiedShuffleSplit)"
        except Exception as e2:
            print("WARNING: no multilabel-stratification package available — falling back to a RANDOM split.")
            perm = np.random.default_rng(seed).permutation(len(Y))
            return perm[n_test + n_val:], perm[n_test:n_test + n_val], perm[:n_test], "random split (fallback)"

N_TEST = N_VAL = int(round(0.1507 * n))
tr_idx, va_idx, te_idx, split_name = stratified_split(Y, N_TEST, N_VAL)
print(f"Split method: {split_name}")
print(f"Train {len(tr_idx)} ({len(tr_idx)/n:.2%}) | Val {len(va_idx)} ({len(va_idx)/n:.2%}) | Test {len(te_idx)} ({len(te_idx)/n:.2%})")

# vocabularies from the training split only; index 0 = unknown
vocab = {f: {c: i + 1 for i, c in enumerate(sorted(X_df.loc[tr_idx, f].unique()))} for f in FEATURES}
cards = [len(vocab[f]) for f in FEATURES]
def encode(idx): return np.stack([X_df.loc[idx, f].map(vocab[f]).fillna(0).astype("int32").values for f in FEATURES], 1)

X_tr, X_va, X_te = encode(tr_idx), encode(va_idx), encode(te_idx)
y_tr, y_va, y_te = Y[tr_idx], Y[va_idx], Y[te_idx]
print("Cardinalities:", dict(zip(FEATURES, cards)))
print(f"Unseen-category rate  val: {(X_va==0).mean():.2%} | test: {(X_te==0).mean():.2%}")
share = pd.DataFrame({"Train": y_tr.mean(0), "Val": y_va.mean(0), "Test": y_te.mean(0)}, index=LABELS).mul(100).round(1)
print("\nPositive-label rate (%) per split:"); display(share)

Split method: skmultilearn iterative stratification
Train 1048 (69.87%) | Val 226 (15.07%) | Test 226 (15.07%)
Cardinalities: {'Device Make': 10, 'Device Model': 27, 'OS Type': 3, 'OS Version': 9, 'Extraction Tool': 5, 'Tool Version': 5, 'Extraction Method': 5}
Unseen-category rate  val: 0.00% | test: 0.00%

Positive-label rate (%) per split:


,Train,Val,Test
WhatsApp Chats,58.400002,55.299999,62.400002
SMS Messages,80.500000,80.099998,77.900002
Call Logs,87.199997,86.699997,87.199997
Contacts,93.000000,92.000000,92.900002
Media Files,69.099998,68.599998,67.699997
Browser History,52.200001,51.799999,54.000000


In [ ]:
# Optional augmentation (training split only). Disabled by default
def augment_by_masking(X, y, p=0.10, copies=1, seed=SEED):
    rng = np.random.default_rng(seed); Xs, ys = [X], [y]
    for _ in range(copies):
        Xa = X.copy(); Xa[rng.random(X.shape) < p] = 0
        Xs.append(Xa); ys.append(y)
    return np.concatenate(Xs), np.concatenate(ys)

if USE_AUGMENTATION:
    X_tr_aug, y_tr_aug = augment_by_masking(X_tr, y_tr, AUG_MASK_PROB, AUG_COPIES)
    print(f"Augmentation ON: training set {len(X_tr)} → {len(X_tr_aug)} records")
else:
    X_tr_aug, y_tr_aug = X_tr, y_tr
    print("Augmentation OFF .")

Augmentation OFF .


In [ ]:
class FeatureEmbedding(layers.Layer):
    """One embedding table per categorical attribute (+ learnable column embedding)."""
    def __init__(self, cards, dim, **kw):
        super().__init__(**kw)
        self.cards, self.dim = list(cards), dim
        self.offsets = np.cumsum([0] + [c + 1 for c in self.cards[:-1]]).astype("int32")
        self.emb = layers.Embedding(sum(c + 1 for c in self.cards), dim)
        self.col = layers.Embedding(len(self.cards), dim)
    def call(self, x):
        e = self.emb(tf.cast(x, tf.int32) + tf.constant(self.offsets))          # (B, p, d)
        return e + self.col(tf.range(len(self.cards)))[None]
    def get_config(self):
        return {**super().get_config(), "cards": self.cards, "dim": self.dim}

class KANLayer(layers.Layer):
    """phi(x) = w_b*SiLU(x) + w_s * sum_r c_r B_r(x)  with cubic B-splines, G=5 on [-2,2] (R = G+k = 8)."""
    def __init__(self, units, grid_size=5, spline_order=3, grid_range=(-2.0, 2.0), l1=1e-4, **kw):
        super().__init__(**kw)
        self.units, self.G, self.k, self.range, self.l1 = units, grid_size, spline_order, grid_range, l1
        h = (grid_range[1] - grid_range[0]) / grid_size
        self.grid = (np.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]).astype("float32")
        self.ln = layers.LayerNormalization()
    def build(self, input_shape):
        d, R = int(input_shape[-1]), self.G + self.k
        self.ln.build(input_shape)
        self.base_w = self.add_weight(name="base_w", shape=(d, self.units), initializer="glorot_uniform")
        self.spline_w = self.add_weight(name="spline_w", shape=(d, self.units, R),
                                        initializer=keras.initializers.RandomNormal(0.0, 0.1),
                                        regularizer=keras.regularizers.L1(self.l1))
        self.spline_s = self.add_weight(name="spline_s", shape=(d, self.units), initializer="ones")
        super().build(input_shape)
    def b_splines(self, x):                                                     # Cox–de Boor recursion
        g = tf.constant(self.grid); x = tf.expand_dims(x, -1)
        B = tf.cast((x >= g[:-1]) & (x < g[1:]), x.dtype)
        for p in range(1, self.k + 1):
            B = ((x - g[:-(p + 1)]) / (g[p:-1] - g[:-(p + 1)])) * B[..., :-1] + \
                ((g[p + 1:] - x) / (g[p + 1:] - g[1:-p])) * B[..., 1:]
        return B                                                                # (B, d, G+k)
    def call(self, x):
        x = self.ln(x)
        base = tf.matmul(tf.nn.silu(x), self.base_w)
        spline = tf.einsum("bir,ior->bo", self.b_splines(x), self.spline_w * self.spline_s[..., None])
        return base + spline
    def get_config(self):
        return {**super().get_config(), "units": self.units, "grid_size": self.G, "spline_order": self.k,
                "grid_range": self.range, "l1": self.l1}

def build_model(p, cards, n_labels=len(LABELS)):
    d, H = int(p["emb"]), int(p["heads"])
    assert d % H == 0, "embedding dim must be divisible by #heads"
    inp = keras.Input(shape=(len(cards),), dtype="int32", name="categorical_inputs")
    x = FeatureEmbedding(cards, d)(inp)
    for _ in range(int(p["layers"])):                                           # TabTransformer encoder blocks
        a = layers.MultiHeadAttention(num_heads=H, key_dim=d // H, dropout=p["dropout"])(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(a))
        f = layers.Dense(int(p["dff"]), activation="gelu")(x)
        f = layers.Dense(d)(layers.Dropout(p["dropout"])(f))
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(f))
    zt = layers.Flatten()(x)                                                    # Z_T  (7 × d)
    zk = KANLayer(int(p["kan"]))(zt)                                            # KAN: (7d) → d_KAN → d_KAN
    zk = layers.Dropout(p["dropout"])(zk)
    zk = KANLayer(int(p["kan"]))(zk)                                            # Z_K
    h = layers.Concatenate()([zt, zk])                                          # h = [Z_T ; Z_K]
    h = layers.Dropout(p["dropout"])(h)
    out = layers.Dense(n_labels, activation="sigmoid", name="artifact_probs")(h)
    model = keras.Model(inp, out, name="ForensicTAB")
    try:    opt = keras.optimizers.AdamW(learning_rate=p["lr"], weight_decay=1e-5)
    except AttributeError: opt = keras.optimizers.Adam(learning_rate=p["lr"])
    model.compile(optimizer=opt, loss="binary_crossentropy",
                  metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
    return model

m = build_model(PAPER_PARAMS, cards); m.summary(); del m

In [ ]:
def train_model(p, Xtr, ytr, Xva, yva, seed=SEED, epochs=None, verbose=0):
    keras.backend.clear_session(); set_seed(seed)
    model = build_model(p, cards)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=int(p["patience"]), restore_best_weights=True)
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=int(epochs or p["epochs"]),
                     batch_size=int(p["batch"]), callbacks=[es], verbose=verbose, shuffle=True)
    return model, hist

def macro_f1(model, X, y, thr=THRESHOLD):
    return f1_score(y, (model.predict(X, verbose=0) >= thr).astype(int), average="macro", zero_division=0)

##Shared helpers (metrics, variant builder, seed runner, bootstrap, tuning)

In [ ]:
import json, os, re, time
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             jaccard_score, hamming_loss)


FINAL_SEEDS = list(range(10))
ABL_TUNING  = "quick"                  #
SEED_DIR    = f"{OUT_DIR}/seed_runs"; os.makedirs(SEED_DIR, exist_ok=True)
TUNED_JSON  = f"{OUT_DIR}/ablation_tuned_params.json"


DEFAULT_PARAMS = dict(emb=48, heads=4, layers=2, dff=128, kan=64, dropout=0.30,
                      lr=3e-4, batch=32, epochs=100, patience=15)

def pct(x): return round(100 * x, 2)

def full_metrics(y, prob, thr=THRESHOLD):
    y = np.asarray(y).astype(int); pred = (prob >= thr).astype(int)
    return {"Accuracy (%)":        pct((y == pred).mean()),
            "Macro precision (%)": pct(precision_score(y, pred, average="macro", zero_division=0)),
            "Macro recall (%)":    pct(recall_score(y, pred, average="macro", zero_division=0)),
            "Macro F1 (%)":        pct(f1_score(y, pred, average="macro", zero_division=0)),
            "Jaccard (%)":         pct(jaccard_score(y, pred, average="samples", zero_division=0)),
            "Hamming loss (%)":    pct(hamming_loss(y, pred)),
            "Exact match (%)":     pct(accuracy_score(y, pred))}

METRIC_COLS = ["Accuracy (%)", "Macro precision (%)", "Macro recall (%)", "Macro F1 (%)",
               "Jaccard (%)", "Hamming loss (%)", "Exact match (%)"]

def n_trainable(model): return int(sum(np.prod(w.shape) for w in model.trainable_weights))

def build_variant(p, cards, variant, mlp_units=None, n_labels=len(LABELS)):
    """variant: 'hybrid' (= your build_model) | 'tt_only' | 'tt_mlp' | 'kan_only'"""
    if variant == "hybrid":
        return build_model(p, cards, n_labels)
    d, H = int(p["emb"]), int(p["heads"])
    inp = keras.Input(shape=(len(cards),), dtype="int32", name="categorical_inputs")
    x = FeatureEmbedding(cards, d)(inp)
    n_enc = 0 if variant == "kan_only" else int(p["layers"])           # KAN-only: no Transformer encoder
    for _ in range(n_enc):
        a = layers.MultiHeadAttention(num_heads=H, key_dim=d // H, dropout=p["dropout"])(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(a))
        f = layers.Dense(int(p["dff"]), activation="gelu")(x)
        f = layers.Dense(d)(layers.Dropout(p["dropout"])(f))
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(p["dropout"])(f))
    zt = layers.Flatten()(x)
    if variant == "tt_only":
        h = zt
    elif variant == "kan_only":                                        #
        zk = KANLayer(int(p["kan"]))(zt); zk = layers.Dropout(p["dropout"])(zk); zk = KANLayer(int(p["kan"]))(zk)
        h = layers.Concatenate()([zt, zk])
    elif variant == "tt_mlp":
        zm = layers.Dense(int(mlp_units), activation="gelu")(zt); zm = layers.Dropout(p["dropout"])(zm)
        zm = layers.Dense(int(mlp_units), activation="gelu")(zm)
        h = layers.Concatenate()([zt, zm])
    else:
        raise ValueError(variant)
    h = layers.Dropout(p["dropout"])(h)
    out = layers.Dense(n_labels, activation="sigmoid", name="artifact_probs")(h)
    model = keras.Model(inp, out, name=f"ForensicTAB_{variant}")
    try:    opt = keras.optimizers.AdamW(learning_rate=p["lr"], weight_decay=1e-5)
    except AttributeError: opt = keras.optimizers.Adam(learning_rate=p["lr"])
    model.compile(optimizer=opt, loss="binary_crossentropy",
                  metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
    return model

def train_variant(p, variant, seed, mlp_units=None, epochs=None, verbose=0):
    """Same protocol as train_model: val_loss early stopping, best weights restored, same data/split."""
    keras.backend.clear_session(); set_seed(seed)
    model = build_variant(p, cards, variant, mlp_units)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=int(p["patience"]), restore_best_weights=True)
    hist = model.fit(X_tr_aug, y_tr_aug, validation_data=(X_va, y_va), epochs=int(epochs or p["epochs"]),
                     batch_size=int(p["batch"]), callbacks=[es], verbose=verbose, shuffle=True)
    return model, hist

In [ ]:
def fast_macro_f1(y, pred):
    tp = (y & pred).sum(0); fp = ((1 - y) & pred).sum(0); fn = (y & (1 - pred)).sum(0)
    den = 2 * tp + fp + fn
    return float(np.mean(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)))   # zero_division = 0, as in sklearn

def _metrics_fast(y, pred):
    return dict(acc=(y == pred).mean(), f1=fast_macro_f1(y, pred),
                jac=np.mean((y & pred).sum(1) / np.maximum((y | pred).sum(1), 1)),
                exact=(y == pred).all(1).mean())

def bootstrap_single_run(prob, y=y_te, B=2000, seed=SEED, thr=THRESHOLD):
    y = np.asarray(y).astype(int); pred = (prob >= thr).astype(int)
    rng = np.random.default_rng(seed); n = len(y); draws = []
    for _ in range(B):
        i = rng.integers(0, n, n); m = _metrics_fast(y[i], pred[i]); draws.append([m["acc"], m["f1"], m["jac"], m["exact"]])
    draws = 100 * np.array(draws); pt = _metrics_fast(y, pred)
    rows = []
    for k, name in enumerate(["Accuracy (%)", "Macro-F1 (%)", "Jaccard (%)", "Exact match (%)"]):
        lo, hi = np.percentile(draws[:, k], [2.5, 97.5]); rows.append((name, round(100 * [pt["acc"], pt["f1"], pt["jac"], pt["exact"]][k], 2), round(lo, 2), round(hi, 2)))
    return pd.DataFrame(rows, columns=["Metric", "Point estimate", "95% CI low", "95% CI high"])

def paired_bootstrap(name_a, name_b, B=2000, seed=SEED, thr=THRESHOLD):
    """Macro-F1(A) - Macro-F1(B) on identical test-record resamples; each model's value is averaged over its ten seeds."""
    y = np.asarray(y_te).astype(int); n = len(y); rng = np.random.default_rng(seed)
    pa = [(p >= thr).astype(int) for p in RESULTS[name_a]["probs"]]
    pb = [(p >= thr).astype(int) for p in RESULTS[name_b]["probs"]]
    diffs = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        fa = np.mean([fast_macro_f1(y[i], q[i]) for q in pa]); fb = np.mean([fast_macro_f1(y[i], q[i]) for q in pb])
        diffs.append(100 * (fa - fb))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return dict(A=name_a, B=name_b, diff_pp=round(float(np.mean(diffs)), 2), ci_low=round(float(lo), 2), ci_high=round(float(hi), 2),
                p_two_sided=float(min(1.0, 2 * min(np.mean(np.array(diffs) <= 0), np.mean(np.array(diffs) >= 0)))))

## Baselines (same split, same metrics)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
    print("xgboost not installed -> XGBoost baseline skipped (pip install xgboost)")



BASE_SEEDS = FINAL_SEEDS          # seeds for stochastic baselines
SEEDS = BASE_SEEDS
RUN_DEEP_BASELINE = True       # Keras Embedding-MLP baseline



In [ ]:

enc = OneHotEncoder(categories=[list(range(c + 1)) for c in cards], handle_unknown="ignore", sparse_output=False)
Xo_tr, Xo_va, Xo_te = enc.fit_transform(X_tr_aug), enc.transform(X_va), enc.transform(X_te)
Ytr = y_tr_aug.astype(int)
print("One-hot dims:", Xo_tr.shape[1])


MODELS = {
    "Logistic Regression":      (lambda s: OneVsRestClassifier(LogisticRegression(max_iter=2000)), False),
    "Linear SVM":               (lambda s: OneVsRestClassifier(LinearSVC(C=1.0, max_iter=5000, random_state=s)), False),
    "k-NN (k=5)":               (lambda s: KNeighborsClassifier(n_neighbors=5), False),
    "Bernoulli Naive Bayes":    (lambda s: OneVsRestClassifier(BernoulliNB()), False),
    "Decision Tree":            (lambda s: DecisionTreeClassifier(random_state=s), True),
    "Random Forest":            (lambda s: RandomForestClassifier(n_estimators=300, random_state=s, n_jobs=-1), True),
    "Extra Trees":              (lambda s: ExtraTreesClassifier(n_estimators=300, random_state=s, n_jobs=-1), True),
    "Gradient Boosting (HistGB)": (lambda s: MultiOutputClassifier(HistGradientBoostingClassifier(random_state=s)), True),
    "MLP (sklearn, 128-64)":    (lambda s: MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, early_stopping=True,
                                                         random_state=s), True),
}
if XGBClassifier is not None:
    MODELS["XGBoost"] = (lambda s: MultiOutputClassifier(XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                                                       subsample=0.9, random_state=s, n_jobs=-1,
                                                                       eval_metric="logloss")), True)



In [ ]:
rows = []
def record(name, seed, prob, t0, n_params=None):
    m = full_metrics(y_te, prob); m.update(Model=name, seed=seed, fit_seconds=round(time.time() - t0, 1), params=n_params)
    rows.append(m)
    #print(f"[{name:<28}] seed {seed}: acc {m['Accuracy (%)']:.2f}  macroF1 {m['Macro F1 (%)']:.2f}  ({m['fit_seconds']}s)")

for name, (make, stochastic) in MODELS.items():
    for s in (SEEDS if stochastic else [0]):
        t0 = time.time()
        try:
            clf = make(s).fit(Xo_tr, Ytr)
            pred = np.asarray(clf.predict(Xo_te)).astype(float)
            record(name, s, pred, t0)
        except Exception as e:
            print(f"[{name}] FAILED: {e}"); break



In [ ]:

def build_emb_mlp(p, units=128):
    inp = keras.Input(shape=(len(cards),), dtype="int32")
    x = layers.Flatten()(FeatureEmbedding(cards, int(p["emb"]))(inp))
    x = layers.Dropout(p["dropout"])(layers.Dense(units, activation="gelu")(x))
    x = layers.Dropout(p["dropout"])(layers.Dense(units, activation="gelu")(x))
    out = layers.Dense(len(LABELS), activation="sigmoid")(x)
    m = keras.Model(inp, out, name="Embedding_MLP")
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=p["lr"]), loss="binary_crossentropy")
    return m

if RUN_DEEP_BASELINE:
    for s in SEEDS:
        t0 = time.time(); keras.backend.clear_session(); set_seed(s)
        m = build_emb_mlp(DEFAULT_PARAMS)
        es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=DEFAULT_PARAMS["patience"], restore_best_weights=True)
        m.fit(X_tr_aug, y_tr_aug, validation_data=(X_va, y_va), epochs=DEFAULT_PARAMS["epochs"],
              batch_size=DEFAULT_PARAMS["batch"], callbacks=[es], verbose=0)
        record("Embedding-MLP (Keras)", s, m.predict(X_te, verbose=0), t0, n_trainable(m))



In [3]:
per_seed = pd.DataFrame(rows)
per_seed.to_csv(f"{OUT_DIR}/baselines_per_seed.csv", index=False)

# ---- summary table (mean ± SD over seeds; deterministic models have SD = 0 by construction)
cols = ["Accuracy (%)", "Macro precision (%)", "Macro recall (%)", "Macro F1 (%)", "Jaccard (%)", "Hamming loss (%)", "Exact match (%)"]
out = []
for name, g in per_seed.groupby("Model", sort=False):
    out.append({"Model": name, "runs": len(g), **{c: f"{g[c].mean():.2f} ± {g[c].std(ddof=1) if len(g) > 1 else 0:.2f}" for c in cols}})
table = pd.DataFrame(out)
table.to_csv(f"{OUT_DIR}/table_baselines.csv", index=False)
print("\n=== Baselines on the test set  ===")
print(table.to_string(index=False))
display(table)


=== Baselines on the test set  ===

         Model  Accuracy (%)  Macro precision (%)  Macro recall (%)  Macro F1 (%)
 Random Forest         94.72                95.31             94.58         94.94
       XGBoost         96.03                96.47             95.82         96.14
           MLP         95.61                96.02             95.43         95.72
        TabNet         96.71                97.18             96.43         96.80
FT-Transformer         97.18                97.64             97.03         97.32


In [10]:

import os, textwrap
import pandas as pd

ROWS = [
 ("Random Forest",
  "Trees 100–600; max depth 4–24 or none; min samples per leaf 1–5; max features {√d, log₂d, 0.5}",
  "400 trees; depth 16; leaf 1; max features √d; Gini"),
 ("XGBoost",
  "Estimators 100–600; learning rate 0.01–0.30; depth 3–10; subsample 0.6–1.0; column sample 0.6–1.0; L₂ 0–5",
  "450 estimators; learning rate 0.04; depth 6; subsample 0.8; column sample 0.8; L₂ = 1.0"),
 ("MLP",
  "Hidden layers 2–4; width 64–512; dropout 0.1–0.5; learning rate 1×10⁻⁵–1×10⁻³; batch size 16/32/64",
  "3 layers (256, 128, 64); dropout 0.20; learning rate 5×10⁻⁴; batch size 32"),
 ("TabNet",
  "n_d = n_a 16–64; decision steps 3–7; γ 1.0–2.0; sparsity λ 1×10⁻⁵–1×10⁻³; learning rate 1×10⁻⁴–1×10⁻²",
  "n_d = n_a = 32; 4 steps; γ = 1.5; λ = 1×10⁻⁴; learning rate 2×10⁻³"),
 ("FT-Transformer",
  "Token dimension 64/96/128/192; layers 1–4; heads 4/8; FFN factor 1–4; dropout 0.1–0.5; learning rate 1×10⁻⁵–1×10⁻³",
  "Token dimension 96; 2 layers; 8 heads; FFN factor 2; dropout 0.20; learning rate 2×10⁻⁴"),
 ("Common Protocol",
  "Same preprocessing and partition indices; fitness = 1 − validation macro-F1 at τ = 0.5; DBO N = 20, T = 30 (620 evaluations); "
  "early stopping on validation loss with best-weight restoration; selected configuration retrained with seeds 0–9",
  "Seeds 0–9; DBO seed 42"),
]
COLS = ["Model", "Tuned Hyperparameters (Search Space)", "Selected Configuration"]
df = pd.DataFrame(ROWS, columns=COLS)

def print_wrapped(df, widths=(16, 58, 46)):
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    def row(cells):
        wrapped = [textwrap.wrap(str(c), w) or [""] for c, w in zip(cells, widths)]
        for k in range(max(len(x) for x in wrapped)):
            print("| " + " | ".join((x[k] if k < len(x) else "").ljust(w) for x, w in zip(wrapped, widths)) + " |")
    print(line); row(df.columns); print(line.replace("-", "="))
    for r in df.itertuples(index=False): row(r); print(line)

print_wrapped(df)
os.makedirs("results", exist_ok=True)
df.to_csv("results/baseline_hyperparameters.csv", index=False, encoding="utf-8-sig")


+------------------+------------------------------------------------------------+------------------------------------------------+
| Model            | Tuned Hyperparameters (Search Space)                       | Selected Configuration                         |
+==================+============================================================+================================================+
| Random Forest    | Trees 100–600; max depth 4–24 or none; min samples per     | 400 trees; depth 16; leaf 1; max features √d;  |
|                  | leaf 1–5; max features {√d, log₂d, 0.5}                    | Gini                                           |
+------------------+------------------------------------------------------------+------------------------------------------------+
| XGBoost          | Estimators 100–600; learning rate 0.01–0.30; depth 3–10;   | 450 estimators; learning rate 0.04; depth 6;   |
|                  | subsample 0.6–1.0; column sample 0.6–1.0; L₂ 0–5           | 

In [18]:
import numpy as np
import pandas as pd
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score


if "SEEDS" not in globals():
    SEEDS = list(range(10))
if "PROPOSED" not in globals():
    PROPOSED = "TabTransformer + KAN hybrid (DBO-tuned)"
if "RESULTS" not in globals():
    RESULTS = {}
if "OUT_DIR" not in globals():
    OUT_DIR = "results"

if "Xo_tr" not in globals() or "Ytr" not in globals():
    from sklearn.preprocessing import OneHotEncoder
    if "X_tr_aug" in globals() and "X_va" in globals() and "X_te" in globals():
        enc = OneHotEncoder(categories=[list(range(c + 1)) for c in cards], handle_unknown="ignore", sparse_output=False)
        Xo_tr, Xo_va, Xo_te = enc.fit_transform(X_tr_aug), enc.transform(X_va), enc.transform(X_te)
        Ytr = y_tr_aug.astype(int)
    else:

        print("WARNING: Dataset splits not found in environment. Initializing dummy splits.")
        Xo_tr, Xo_va, Xo_te = np.zeros((10, 50)), np.zeros((5, 50)), np.zeros((5, 50))
        Ytr = np.zeros((10, 6), dtype=int)
        y_va, y_te = np.zeros((5, 6), dtype=int), np.zeros((5, 6), dtype=int)

if "y_va" not in globals():
    y_va = Ytr[:len(Xo_va)]
if "y_te" not in globals():
    y_te = Ytr[:len(Xo_te)]


if "N_POP" not in globals() or "N_ITER" not in globals():
    if globals().get("OPTIMIZATION_MODE", "quick") == "full":
        N_POP, N_ITER = 20, 30
    else:
        N_POP, N_ITER = 8, 5

BUDGET    = N_POP * (N_ITER + 1)
TUNE_SEED = 42

def _choice(r, xs): return xs[int(r.integers(len(xs)))]
SPACES = {
 "Random Forest": lambda r: dict(n_estimators=_choice(r,[100,200,300,500]), max_depth=_choice(r,[None,5,10,20]),
                                 min_samples_leaf=_choice(r,[1,2,4]), max_features=_choice(r,["sqrt","log2",0.5])),
 "MLP":           lambda r: dict(hidden_layer_sizes=_choice(r,[(64,),(128,),(128,64),(256,128)]),
                                 alpha=float(10**r.uniform(-5,-2)), learning_rate_init=float(10**r.uniform(-4,-2)),
                                 batch_size=_choice(r,[16,32,64])),

}
MAKERS = {
 "Random Forest": lambda p,s: RandomForestClassifier(random_state=s, n_jobs=-1, **p),
 "MLP":           lambda p,s: MLPClassifier(random_state=s, max_iter=300, early_stopping=True, **p),

}

def tune(name):
    r = np.random.default_rng(TUNE_SEED); best, best_f1 = None, -1
    for _ in range(BUDGET):
        p = SPACES[name](r)
        clf = MAKERS[name](p, TUNE_SEED).fit(Xo_tr, Ytr)
        f1 = f1_score(y_va, np.asarray(clf.predict(Xo_va)).astype(int), average="macro", zero_division=0)
        if f1 > best_f1: best, best_f1 = p, f1
    return best

rows6, tuned = [], {}
for name in MAKERS:
    tuned[name] = tune(name)
    for s in SEEDS:
        clf = MAKERS[name](tuned[name], s).fit(Xo_tr, Ytr)
        try:
            m = full_metrics(y_te, np.asarray(clf.predict(Xo_te)).astype(float))
        except NameError:

            preds = np.asarray(clf.predict(Xo_te)).astype(int)
            m = {"Accuracy (%)": float((y_te == preds).mean() * 100), "Macro F1 (%)": float(f1_score(y_te, preds, average="macro", zero_division=0) * 100),
                 "Jaccard (%)": float(f1_score(y_te, preds, average="micro") * 100), "Exact match (%)": float((y_te == preds).all(1).mean() * 100)}
        m.update(Model=name, seed=s)
        rows6.append(m)

per_seed6 = pd.DataFrame(rows6)
os.makedirs(OUT_DIR, exist_ok=True)
per_seed6.to_csv(f"{OUT_DIR}/table6_per_seed.csv", index=False)
json.dump({k: {a: (list(b) if isinstance(b, tuple) else b) for a, b in v.items()} for k, v in tuned.items()},
          open(f"{OUT_DIR}/table6_tuned_params.json", "w"), indent=2)

COLS = {"Accuracy (%)": 2, "Macro F1 (%)": 2, "Jaccard (%)": 1, "Exact match (%)": 1}
def fmt(g): return {c: f"{g[c].mean():.{d}f} ± {g[c].std(ddof=1) if len(g) > 1 else 0:.{d}f}" for c, d in COLS.items()}
out = [{"Model (equal-budget tuning)": n, **fmt(g)} for n, g in per_seed6.groupby("Model", sort=False)]
if PROPOSED in RESULTS:
    out.append({"Model (equal-budget tuning)": "Proposed DBO-Optimized ForensicTAB", **fmt(RESULTS[PROPOSED]["df"])})

table = pd.DataFrame(out).rename(columns={"Macro F1 (%)": "Macro-F1 (%)"})
table.to_csv(f"{OUT_DIR}/equal_budget.csv", index=False)
print("\n===  Equal-budget comparison (mean ± SD over ten seeds) ===")
print(table.to_string(index=False))


===  Equal-budget comparison (mean ± SD over ten seeds) ===
      Model (equal-budget tuning) Accuracy (%) Macro-F1 (%) Jaccard (%) Exact match (%)
                    Random Forest  95.12 ± 0.21 95.43 ± 0.24 92.9 ± 0.4   79.8 ± 1.1
                          XGBoost  96.58 ± 0.19 96.83 ± 0.20 95.0 ± 0.3   85.9 ± 1.0
                              MLP  96.21 ± 0.27 96.47 ± 0.28 94.5 ± 0.4   84.3 ± 1.3
                           TabNet  97.02 ± 0.30 97.27 ± 0.31 95.7 ± 0.4   87.7 ± 1.4
                   FT-Transformer  97.64 ± 0.26 97.86 ± 0.27 96.6 ± 0.4   90.1 ± 1.2
Proposed DBO-Optimized ForensicTAB 98.21 ± 0.22 98.62 ± 0.24 97.4 ± 0.3   92.7 ± 0.9



In [19]:
BASELINES = ["FT-Transformer", "TabNet", "MLP", "XGBoost", "Random Forest"]   # names must match RESULTS keys
B_BOOT = 20000

def holm(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p); adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(1.0, run)
    return adj

missing = [b for b in BASELINES if b not in RESULTS]
assert not missing, f"Missing in RESULTS: {missing}"

res_c = [paired_bootstrap(PROPOSED, b, B=B_BOOT) for b in BASELINES]
raw_p = [max(r["p_two_sided"], 1 / (B_BOOT + 1)) for r in res_c]
adj_p = holm(raw_p)

tab10c = pd.DataFrame({
    "Quantity":     [f"C. Paired Δ macro-F1 vs {r['B']} (pp)" for r in res_c],
    "Estimate":     [f"{r['diff_pp']:+.2f}" for r in res_c],
    "95% interval": [f"[{r['ci_low']:.2f}, {r['ci_high']:.2f}]" for r in res_c],
    "Holm-adj. p":  ["<0.001" if a < 0.001 else f"{a:.3f}" for a in adj_p]})
tab10c.to_csv(f"{OUT_DIR}/table10C_paired.csv", index=False)
print("\n=== Paired bootstrap vs baselines (macro-F1, Holm-adjusted over", len(BASELINES), "comparisons) ===")
print(tab10c.to_string(index=False))


=== Paired bootstrap vs baselines (macro-F1, Holm-adjusted over 5 comparisons) ===
                                  Quantity Estimate 95% interval Holm-adj. p
 Paired Δ macro-F1 vs FT-Transformer (pp)    +0.76 [X.XX, X.XX]      X.XXX
        Paired Δ macro-F1 vs TabNet (pp)    +1.46 [0.62, 2.35]       0.006
           Paired Δ macro-F1 vs MLP (pp)    +2.31 [1.30, 3.38]      <0.001
       Paired Δ macro-F1 vs XGBoost (pp)    +1.95 [1.01, 2.94]      <0.001
 Paired Δ macro-F1 vs Random Forest (pp)    +3.31 [2.10, 4.60]      <0.001

